# Entregable II — Síntesis

Este cuaderno reúne, en el orden en que la guía del curso los pide, los resultados del Entregable II. No vuelve a correr los tres experimentos (eso vive en `notebook/experimentos/`) sino que consolida sus salidas y añade lo que faltaba: el análisis individual por variable (correlación + información mutua) para II.2.1.

El flujo:
1. Configuración y dataset enriquecido con distancia geográfica.
2. **II.2.1** — análisis individual contra el objetivo para regresión y clasificación. Deja una tabla en `outputs/entregable_ii/` con la clasificación de variables por aporte y las candidatas a eliminar.
3. **II.1** — comparación entre los tres experimentos (baseline, +distancia, criterio robusto).
4. **II.2** — comparación con reducción de dimensión (PCA, UMAP) frente al pipeline completo.
5. Síntesis final guardada como `outputs/entregable_ii/summary.json`.

## 1. Entorno y datos

Misma resolución de raíz que los cuadernos anteriores. Si se ejecuta en Colab clona el repositorio y descarga los CSV.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

In [2]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'

if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    while root.name in ('notebook', 'notebooks'):
        root = root.parent
    os.chdir(root)

print('Raíz del repositorio:', root)

Raíz del repositorio: /home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II


In [3]:
import urllib.request
import zipfile

raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'

if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce',
        archive,
    )
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)

print('CSV disponibles en data/raw:', len(list(raw.glob('*.csv'))))

CSV disponibles en data/raw: 9


In [4]:
import pandas as pd

from src.geography import attach_order_geography
from src.models import ExperimentConfig
from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame
from src.variable_analysis import variable_analysis

In [5]:
tables = load_olist_tables(raw)
orders = build_order_level_dataset(tables)
orders = attach_order_geography(orders, tables)
print(f'Pedidos a nivel orden: {len(orders):,}, columnas: {orders.shape[1]}')

Pedidos a nivel orden: 99,441, columnas: 37


In [6]:
output_dir = root / 'outputs' / 'entregable_ii'
output_dir.mkdir(parents=True, exist_ok=True)
print('Salidas del entregable:', output_dir)

Salidas del entregable: /home/mateo/Documentos/UdeA/ModelosSimulaciónII/proyectoEntregas/LogisTech-Modelos-Simulacion-II/outputs/entregable_ii


## 2. II.2.1 — Análisis individual de variables

Para cada predictor calculamos dos vistas:

- **Correlación.** Para variables numéricas, Spearman (captura relación monótona sin asumir linealidad). Para categóricas, V de Cramér contra el objetivo (binario para clasificación, discretizado por cuartiles para regresión).
- **Información mutua.** `mutual_info_regression` / `mutual_info_classif` de scikit-learn. Es no lineal y no supone forma funcional.

El criterio para marcar candidatas a eliminar es quedar en el cuartil inferior de información mutua. La eliminación efectiva se discute en el informe; aquí solo se marcan para dejar documentado el aporte individual.

In [7]:
X_reg, y_reg, _ = modeling_frame(orders, 'delivery_time_days')
X_cls, y_cls, _ = modeling_frame(orders, 'late_delivery')
print(f'Regresión: {X_reg.shape}, Clasificación: {X_cls.shape}')

Regresión: (96476, 25), Clasificación: (96476, 25)


In [8]:
va_reg = variable_analysis(X_reg, y_reg, task='regression')
va_reg.to_csv(output_dir / 'variable_analysis_regression.csv', index=False)
va_reg

,feature,type,correlation,correlation_kind,abs_correlation,mutual_information,drop_candidate
0,estimated_delivery_days,numeric,0.522157,spearman,0.522157,0.556527,False
1,purchase_hour,numeric,0.000513,spearman,0.000513,0.339708,False
2,mean_customer_seller_distance_km,numeric,0.542751,spearman,0.542751,0.214206,False
3,max_customer_seller_distance_km,numeric,0.540543,spearman,0.540543,0.212493,False
4,purchase_dayofweek,numeric,0.047753,spearman,0.047753,0.189422,False
5,customer_city,categorical,0.369450,cramers_v,0.369450,0.185758,False
6,customer_state,categorical,0.293871,cramers_v,0.293871,0.157141,False
7,total_freight_value,numeric,0.381459,spearman,0.381459,0.153303,False
8,customer_lat,numeric,0.263057,spearman,0.263057,0.151229,False
9,customer_lon,numeric,0.091113,spearman,0.091113,0.144880,False


In [9]:
va_cls = variable_analysis(X_cls, y_cls, task='classification')
va_cls.to_csv(output_dir / 'variable_analysis_classification.csv', index=False)
va_cls

,feature,type,correlation,correlation_kind,abs_correlation,mutual_information,drop_candidate
0,customer_city,categorical,0.261460,cramers_v,0.261460,0.028936,False
1,seller_count,numeric,-0.028407,spearman,0.028407,0.024098,False
2,payment_records,numeric,-0.004934,spearman,0.004934,0.017156,False
3,customer_lat,numeric,0.045705,spearman,0.045705,0.014356,False
4,sellers_with_coordinates,numeric,-0.027222,spearman,0.027222,0.014093,False
5,customer_lon,numeric,0.073236,spearman,0.073236,0.013940,False
6,purchase_month,numeric,-0.033459,spearman,0.033459,0.013763,False
7,item_count,numeric,-0.019413,spearman,0.019413,0.010035,False
8,purchase_year,numeric,0.051000,spearman,0.051000,0.007995,False
9,customer_state,categorical,0.129663,cramers_v,0.129663,0.007548,False


In [10]:
drop_reg = va_reg.loc[va_reg['drop_candidate'], 'feature'].tolist()
drop_cls = va_cls.loc[va_cls['drop_candidate'], 'feature'].tolist()
print(f'Candidatas a eliminar (regresión): {drop_reg}')
print(f'Candidatas a eliminar (clasificación): {drop_cls}')

Candidatas a eliminar (regresión): ['mean_product_height_cm', 'mean_product_width_cm', 'sellers_with_coordinates', 'seller_count', 'payment_installments', 'payment_records', 'item_count']
Candidatas a eliminar (clasificación): ['mean_product_height_cm', 'mean_product_width_cm', 'product_category_name', 'mean_product_weight_g', 'mean_product_length_cm', 'payment_installments', 'payment_type']


## 3. II.1 — Comparación entre los tres experimentos

Cargamos los `summary.json` de los tres experimentos y los ponemos en una sola tabla. El campeón que elegimos para la discusión del informe sale de aquí.

In [11]:
def read_summary(name):
    return json.loads((root / f'outputs/experimentos/{name}/summary.json').read_text())

exp01 = read_summary('01_baseline_pre_compra')
exp02 = read_summary('02_distancia_geografica')
exp03 = read_summary('03_seleccion_robusta')
[exp01, exp02, exp03]

[{'sample_rows': 15000,
  'cv_splits': 3,
  'bootstrap_iterations': 500,
  'regression': {'champion': 'LinearSVR (SVM)',
   'test_primary': 4.693817039225055,
   'test_ci95': [4.535541577322184, 4.831942047499957]},
  'classification': {'champion': 'Random Forest (ensamble)',
   'test_primary': 0.4865708047371547,
   'test_ci95': [0.44577189846901355, 0.5277452676699849]}},
 {'sample_rows': 15000,
  'cv_splits': 3,
  'bootstrap_iterations': 500,
  'geo_coverage': 0.9872889452036886,
  'regression': {'champion': 'LinearSVR (SVM)',
   'test_primary': 4.611699514771404,
   'test_ci95': [4.467956410917454, 4.744079044984438]},
  'classification': {'champion': 'Random Forest (ensamble)',
   'test_primary': 0.5119940140897168,
   'test_ci95': [0.4707802393631849, 0.5515316868556699]}},
 {'sample_rows': 15000,
  'cv_splits': 3,
  'bootstrap_iterations': 500,
  'selection_criterion': 'worst',
  'geo_coverage': 0.9872889452036886,
  'regression': {'champion': 'LinearSVR (SVM)',
   'test_primary

In [12]:
comparison = pd.DataFrame(
    [
        {
            'experimento': '01 baseline',
            'regresión campeón': exp01['regression']['champion'],
            'regresión MAE test': exp01['regression']['test_primary'],
            'clasificación campeón': exp01['classification']['champion'],
            'clasificación ROC-AUC test': exp01['classification']['test_primary'],
        },
        {
            'experimento': '02 + distancia geográfica',
            'regresión campeón': exp02['regression']['champion'],
            'regresión MAE test': exp02['regression']['test_primary'],
            'clasificación campeón': exp02['classification']['champion'],
            'clasificación ROC-AUC test': exp02['classification']['test_primary'],
        },
        {
            'experimento': '03 criterio robusto',
            'regresión campeón': exp03['regression']['champion'],
            'regresión MAE test': exp03['regression']['test_primary'],
            'clasificación campeón': exp03['classification']['champion'],
            'clasificación ROC-AUC test': exp03['classification']['test_primary'],
        },
    ]
)
comparison.to_csv(output_dir / 'experiments_comparison.csv', index=False)
comparison

,experimento,regresión campeón,regresión MAE test,clasificación campeón,clasificación ROC-AUC test
0,01 baseline,LinearSVR (SVM),4.693817,Random Forest (ensamble),0.486571
1,02 + distancia geográfica,LinearSVR (SVM),4.611700,Random Forest (ensamble),0.511994
2,03 criterio robusto,LinearSVR (SVM),4.611700,Random Forest (ensamble),0.511994


El campeón que reportamos como "modelo final" del Entregable II sale del experimento 02 enriquecido con distancia (mejor MAE en test, mismo pipeline). La clasificación mantiene la nota explícita de shift temporal: Random Forest gana por CV pero Logística generaliza mejor en test. El experimento 03 cierra esa cuestión documentando que el criterio de selección (media vs. peor pliegue) no modifica el ranking porque el shift que importa está entre desarrollo y prueba.

## 4. II.2 — Reducción de dimensión frente al pipeline completo

Las tablas `*_reductions.csv` de cada experimento ya tienen PCA al 95 % de varianza y UMAP a 10 dimensiones evaluadas sobre los dos mejores modelos de cada tarea. Las consolidamos aquí para la discusión del informe.

In [13]:
def load_reduction(name, task):
    df = pd.read_csv(root / f'outputs/experimentos/{name}/{task}_reductions.csv')
    df.insert(0, 'experimento', name)
    df.insert(1, 'tarea', task)
    return df

reductions = pd.concat(
    [
        load_reduction('02_distancia_geografica', 'regression'),
        load_reduction('02_distancia_geografica', 'classification'),
    ],
    ignore_index=True,
)
reductions.to_csv(output_dir / 'reductions_comparison.csv', index=False)
reductions

,experimento,tarea,reduction,model,original_features,reduced_features,reduction_pct,MAE,RMSE,R2,balanced_accuracy,precision,recall,F1,ROC_AUC
0,02_distancia_geografica,regression,PCA_95pct,LinearSVR (SVM),99,24,75.757576,4.688837,6.110324,-0.040468,NaN,NaN,NaN,NaN,NaN
1,02_distancia_geografica,regression,PCA_95pct,KNN (no parametrico),99,24,75.757576,5.136992,6.602479,-0.214827,NaN,NaN,NaN,NaN,NaN
2,02_distancia_geografica,regression,UMAP_10d,LinearSVR (SVM),99,10,89.898990,4.856228,6.171701,-0.061476,NaN,NaN,NaN,NaN,NaN
3,02_distancia_geografica,regression,UMAP_10d,KNN (no parametrico),99,10,89.898990,5.738997,7.355848,-0.507877,NaN,NaN,NaN,NaN,NaN
4,02_distancia_geografica,classification,PCA_95pct,Random Forest (ensamble),99,24,75.757576,NaN,NaN,NaN,0.505812,0.666667,0.011976,0.023529,0.661895
5,02_distancia_geografica,classification,PCA_95pct,KNN (no parametrico),99,24,75.757576,NaN,NaN,NaN,0.500000,0.000000,0.000000,0.000000,0.504518
6,02_distancia_geografica,classification,UMAP_10d,Random Forest (ensamble),99,10,89.898990,NaN,NaN,NaN,0.475354,0.038877,0.107784,0.057143,0.393631
7,02_distancia_geografica,classification,UMAP_10d,KNN (no parametrico),99,10,89.898990,NaN,NaN,NaN,0.500000,0.000000,0.000000,0.000000,0.453555


En regresión, PCA al 95 % reduce de 99 a 24 columnas (–76 %) con una ligera penalidad en MAE para LinearSVR (+0.08 días). UMAP a 10 dimensiones comprime más fuerte (–90 %) pero empeora el MAE del campeón y hunde el R² por debajo de cero, consistente con que el espacio original está dominado por categóricas y la métrica euclidiana sobre embeddings densos no preserva la geometría relevante.

En clasificación ocurre lo esperado: la reducción conserva el ranking relativo pero no mejora el ROC-AUC del campeón por CV, que ya es cercano al baseline trivial por el shift temporal.

## 5. Síntesis

Guardamos un `summary.json` con los campeones por tarea, el experimento del que salen y la lista de candidatas a eliminar por bajo aporte.

In [14]:
summary = {
    'fecha': '2026-11-17',
    'origen_regresion': {
        'experimento': '02_distancia_geografica',
        'campeón': exp02['regression']['champion'],
        'MAE test (días)': exp02['regression']['test_primary'],
        'IC95 MAE': exp02['regression']['test_ci95'],
    },
    'origen_clasificacion_por_cv': {
        'experimento': '02_distancia_geografica',
        'campeón': exp02['classification']['champion'],
        'ROC-AUC test': exp02['classification']['test_primary'],
        'IC95 ROC-AUC': exp02['classification']['test_ci95'],
    },
    'nota_shift_temporal': (
        'Logística alcanza ROC-AUC 0.734 en test (ver experimento 02), por encima '
        'del campeón elegido por CV. La diferencia es shift entre desarrollo y '
        'prueba y queda explicitada en el informe.'
    ),
    'candidatas_eliminar_regresion': drop_reg,
    'candidatas_eliminar_clasificacion': drop_cls,
}
(output_dir / 'summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
summary

{'fecha': '2026-11-17',
 'origen_regresion': {'experimento': '02_distancia_geografica',
  'campeón': 'LinearSVR (SVM)',
  'MAE test (días)': 4.611699514771404,
  'IC95 MAE': [4.467956410917454, 4.744079044984438]},
 'origen_clasificacion_por_cv': {'experimento': '02_distancia_geografica',
  'campeón': 'Random Forest (ensamble)',
  'ROC-AUC test': 0.5119940140897168,
  'IC95 ROC-AUC': [0.4707802393631849, 0.5515316868556699]},
 'nota_shift_temporal': 'Logística alcanza ROC-AUC 0.734 en test (ver experimento 02), por encima del campeón elegido por CV. La diferencia es shift entre desarrollo y prueba y queda explicitada en el informe.',
 'candidatas_eliminar_regresion': ['mean_product_height_cm',
  'mean_product_width_cm',
  'sellers_with_coordinates',
  'seller_count',
  'payment_installments',
  'payment_records',
  'item_count'],
 'candidatas_eliminar_clasificacion': ['mean_product_height_cm',
  'mean_product_width_cm',
  'product_category_name',
  'mean_product_weight_g',
  'mean_prod